# Vuln2Action - CVE-only target_steps, LR 2e-5, 3 epochs

Muc tieu: kiem tra train collapse la do LR 5e-5 qua cao hay do du lieu.

- target_steps, CVE-only
- lr 2e-5, warmup 0.06
- 3 epochs, eval moi epoch
- batch 8/GPU, grad accum 4
- evaluate 1.000 validation rows cho tung checkpoint


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
hits=sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits: raise FileNotFoundError("flow_train.jsonl not found")
FLOW_DIR=Path(hits[0]).parent
os.environ["FLOW_DIR"]=str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"]="false"
print("FLOW_DIR:", FLOW_DIR)


In [ ]:
import os, sys, subprocess
cmd=[
 sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
 "scripts/train_t5.py",
 "--dataset", os.environ["FLOW_DIR"],
 "--output-dir", "/kaggle/working/t5_stage1_target_steps_lr2e5_3ep",
 "--model-name", "t5-base",
 "--target-field", "target_steps",
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--per-device-batch-size", "8",
 "--gradient-accumulation-steps", "4",
 "--learning-rate", "2e-5",
 "--warmup-ratio", "0.06",
 "--num-train-epochs", "3",
 "--save-steps", "693",
 "--eval-steps", "693",
 "--logging-steps", "100",
 "--save-total-limit", "3",
 "--no-load-best-model-at-end",
 "--fp16",
 "--gradient-checkpointing",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
from pathlib import Path
base=Path("/kaggle/working/t5_stage1_target_steps_lr2e5_3ep")
ckpts=sorted(base.glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]))
print("final exists:", (base/"final").exists())
print("checkpoints:", [p.name for p in ckpts])
if not ckpts: raise FileNotFoundError("no checkpoints")


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
base=Path("/kaggle/working/t5_stage1_target_steps_lr2e5_3ep")
ckpts=sorted(base.glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]))
print("evaluating:", [p.name for p in ckpts])
for ckpt in ckpts:
    out=Path(f"/kaggle/working/eval_{ckpt.name}")
    cmd=[
     sys.executable, "scripts/evaluate_t5.py",
     "--checkpoint", str(ckpt),
     "--dataset", os.environ["FLOW_DIR"],
     "--split", "validation",
     "--target-field", "target_steps",
     "--output-dir", str(out),
     "--max-input-length", "256",
     "--max-target-length", "384",
     "--batch-size", "4",
     "--num-beams", "4",
     "--limit", "1000",
     "--fp16",
    ]
    print("\n=== EVAL", ckpt.name, "===")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
import json
from pathlib import Path
for out in sorted(Path("/kaggle/working").glob("eval_checkpoint-*")):
    m=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
    print(out.name, json.dumps({k:m[k] for k in ["bleu","bleu_corpus","rouge1_f1","rouge2_f1","rougeL_f1"]}, ensure_ascii=False))
